#Preprocessing


In [3]:
"""
Final DDX-Plus Preprocessing Pipeline
Complete workflow: Explore → Map → Process
"""

# ========================================
# 1. SETUP & INSTALLATION
# ========================================

!pip install -q pandas tqdm datasets

import json
import pandas as pd
import ast
from pathlib import Path
from tqdm.notebook import tqdm
import os
from datasets import load_from_disk
from collections import Counter, defaultdict
from typing import Dict, List, Any, Optional
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully!")


✅ Libraries imported successfully!


✅ Libraries imported successfully!
Mounted at /content/drive

✅ Google Drive mounted!

📁 Directory Setup:
   Base: /content/drive/MyDrive/DDX
   Raw data: /content/drive/MyDrive/DDX/raw/ddxplus_hf
   Processed: /content/drive/MyDrive/DDX/processed
   Reports: /content/drive/MyDrive/DDX/reports

🔍 Checking dataset structure...

📂 Contents of /content/drive/MyDrive/DDX/raw/ddxplus_hf:
   📄 dataset_dict.json
   📁 train
   📁 test
   📁 validate

✅ Found 3 splits: ['train', 'test', 'validate']

📥 Loading dataset from disk...
✅ Dataset loaded successfully!

📊 Dataset Info:
   Available splits: ['train', 'test', 'validate']
   • train: 1,025,602 samples
   • test: 134,529 samples
   • validate: 132,448 samples

🚀 STARTING PREPROCESSING PIPELINE

⚙️ Configuration:
  Processing mode: Sample
  Sample size: 1,000 records per split
  Exploration sample: 1,000 records

🏥 DDX-PLUS SMART PREPROCESSING PIPELINE

📊 STEP 1: Exploring Dataset Structure
─────────────────────────────────────────────────────

Processing train:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing validate:   0%|          | 0/1000 [00:00<?, ?it/s]


📊 DATASET STATISTICS

TRAIN SET:
  Total samples: 1,000
  Unique diseases: 47
  Avg symptoms: 0.0 per patient
  Age range: 1-109 years (avg: 39.5)
  Sex distribution:
    • M: 500 (50.0%)
    • F: 500 (50.0%)

  Top 5 diseases:
    • Viral pharyngitis: 73 (7.3%)
    • URTI: 52 (5.2%)
    • Allergic sinusitis: 45 (4.5%)
    • Anemia: 42 (4.2%)
    • Anaphylaxis: 31 (3.1%)

TEST SET:
  Total samples: 1,000
  Unique diseases: 47
  Avg symptoms: 0.0 per patient
  Age range: 1-109 years (avg: 39.4)
  Sex distribution:
    • F: 515 (51.5%)
    • M: 485 (48.5%)

  Top 5 diseases:
    • Viral pharyngitis: 71 (7.1%)
    • URTI: 48 (4.8%)
    • Anemia: 43 (4.3%)
    • Pulmonary embolism: 36 (3.6%)
    • Panic attack: 36 (3.6%)

VALIDATE SET:
  Total samples: 1,000
  Unique diseases: 48
  Avg symptoms: 0.0 per patient
  Age range: 1-104 years (avg: 41.3)
  Sex distribution:
    • F: 531 (53.1%)
    • M: 469 (46.9%)

  Top 5 diseases:
    • URTI: 65 (6.5%)
    • Viral pharyngitis: 60 (6.0%)
    •

In [8]:
# ========================================
# 6. EVIDENCE MAPPER
# ========================================

class EvidenceMapper:
    """Maps evidence codes to human-readable text"""

    def __init__(self):
        self.code_to_text = {}
        logger.info("✅ Evidence mapper initialized")

    def get_text(self, code):
        """Convert evidence code to readable text"""
        if code in self.code_to_text:
            return self.code_to_text[code]

        # Clean and format code
        code_clean = str(code).replace('_', ' ').replace('-', ' ')
        code_clean = code_clean.replace('E ', '').replace('e ', '')
        code_clean = ' '.join(word.capitalize() for word in code_clean.split())

        return code_clean if code_clean else str(code)


In [9]:
# ========================================
# 7. DATA EXPLORER
# ========================================

class DatasetExplorer:
    """Explore dataset structure before processing"""

    def __init__(self, dataset):
        self.dataset = dataset
        self.report = {}

    def explore_split(self, split_name: str, sample_size: int = 1000) -> Dict:
        """Explore a single split thoroughly"""
        logger.info(f"\n{'='*60}")
        logger.info(f"🔍 EXPLORING: {split_name.upper()}")
        logger.info(f"{'='*60}")

        split_data = self.dataset[split_name]
        total_records = len(split_data)
        sample_size = min(sample_size, total_records)

        logger.info(f"📊 Sampling {sample_size:,} of {total_records:,} records...")

        # Convert to pandas for analysis
        df = pd.DataFrame([split_data[i] for i in range(sample_size)])

        report = {
            'total_records': total_records,
            'sampled_records': sample_size,
            'columns': {},
            'data_quality': {}
        }

        # Analyze each column
        for col in df.columns:
            col_info = self._analyze_column(df[col], col)
            report['columns'][col] = col_info

        # Check data quality
        report['data_quality'] = self._check_quality(df)

        return report

    def _analyze_column(self, series: pd.Series, col_name: str) -> Dict:
        """Analyze a single column"""
        logger.info(f"\n📋 Column: {col_name}")

        info = {
            'name': col_name,
            'dtype': str(series.dtype),
            'total': len(series),
            'non_null': int(series.notna().sum()),
            'null': int(series  .isna().sum()),
            'null_pct': round((series.isna().sum() / len(series)) * 100, 2),
            'unique': int(series.nunique()),
        }

        logger.info(f"   Type: {info['dtype']}")
        logger.info(f"   Non-null: {info['non_null']:,} ({100-info['null_pct']:.1f}%)")
        logger.info(f"   Unique: {info['unique']:,}")

        # Analyze non-null values
        non_null = series.dropna()
        if len(non_null) > 0:
            first_val = non_null.iloc[0]
            info['python_type'] = type(first_val).__name__

            # Check for nested structures
            if isinstance(first_val, (dict, list)):
                info['is_nested'] = True
                info['nested_type'] = type(first_val).__name__
                logger.info(f"   ⚠️ Nested structure: {info['nested_type']}")

                if isinstance(first_val, dict):
                    logger.info(f"   Dict keys: {len(first_val)}")
                elif isinstance(first_val, list):
                    logger.info(f"   List length: {len(first_val)}")

                # Show sample
                sample_str = str(first_val)
                if len(sample_str) > 100:
                    sample_str = sample_str[:100] + "..."
                logger.info(f"   Sample: {sample_str}")

            elif isinstance(first_val, str):
                # Check if parseable
                try:
                    parsed = ast.literal_eval(first_val)
                    info['is_nested'] = True
                    info['nested_type'] = f"string_{type(parsed).__name__}"
                    logger.info(f"   ⚠️ String contains: {type(parsed).__name__}")
                except:
                    info['is_nested'] = False
                    # Show value distribution for small unique counts
                    if info['unique'] < 20:
                        logger.info(f"   Values: {series.value_counts().head(5).to_dict()}")
            else:
                # Numeric or other
                info['is_nested'] = False
                if pd.api.types.is_numeric_dtype(series):
                    info['min'] = float(series.min())
                    info['max'] = float(series.max())
                    info['mean'] = float(series.mean())
                    logger.info(f"   Range: {info['min']:.1f} - {info['max']:.1f}")

        return info

    def _check_quality(self, df: pd.DataFrame) -> Dict:
        """Check data quality issues"""
        logger.info(f"\n🔍 Data Quality Checks:")

        issues = {
            'high_missing': {},
            'inconsistent_types': {},
            'empty_nested': 0
        }

        # High missing rate
        for col in df.columns:
            null_pct = (df[col].isna().sum() / len(df)) * 100
            if null_pct > 10:
                issues['high_missing'][col] = f"{null_pct:.1f}%"

        if issues['high_missing']:
            logger.info(f"   ⚠️ High missing values:")
            for col, pct in issues['high_missing'].items():
                logger.info(f"      • {col}: {pct}")
        else:
            logger.info(f"   ✅ No high missing values")

        # Type consistency
        for col in df.select_dtypes(include=['object']).columns:
            types = df[col].dropna().apply(type).unique()
            if len(types) > 1:
                issues['inconsistent_types'][col] = [t.__name__ for t in types]

        if issues['inconsistent_types']:
            logger.info(f"   ⚠️ Inconsistent types:")
            for col, types in issues['inconsistent_types'].items():
                logger.info(f"      • {col}: {types}")

        return issues

    def explore_all(self, sample_size: int = 1000) -> Dict:
        """Explore all splits"""
        logger.info("\n" + "="*70)
        logger.info("🔬 DATASET EXPLORATION")
        logger.info("="*70)

        full_report = {}

        for split_name in self.dataset.keys():
            report = self.explore_split(split_name, sample_size)
            full_report[split_name] = report

        # Save report
        self._save_report(full_report)

        return full_report

    def _save_report(self, report: Dict):
        """Save exploration report"""
        output_path = REPORTS_DIR / "exploration_report.json"
        with open(output_path, 'w') as f:
            json.dump(report, f, indent=2, default=str)
        logger.info(f"\n💾 Exploration report saved: {output_path}")

# ========================================

In [10]:

# 8. COLUMN MAPPER
# ========================================

class ColumnMapper:
    """Map dataset columns to standard schema"""

    def __init__(self, exploration_report: Dict):
        self.report = exploration_report
        self.mapping = self._detect_columns()

    def _detect_columns(self) -> Dict[str, str]:
        """Auto-detect column mapping"""
        logger.info(f"\n🗺️ Detecting column mapping...")

        mapping = {}

        # Get first split's columns
        first_split = list(self.report.keys())[0]
        columns = self.report[first_split]['columns']

        # Detection logic
        for col_name in columns.keys():
            col_lower = col_name.lower()

            if 'evidence' in col_lower or 'symptom' in col_lower:
                mapping['symptoms'] = col_name
            elif 'pathology' in col_lower or 'diagnosis' in col_lower:
                mapping['diagnosis'] = col_name
            elif 'differential' in col_lower:
                mapping['differential'] = col_name
            elif col_lower == 'age':
                mapping['age'] = col_name
            elif col_lower in ['sex', 'gender']:
                mapping['sex'] = col_name

        logger.info(f"   Detected mappings:")
        for standard, actual in mapping.items():
            logger.info(f"      {standard} → {actual}")

        return mapping

    def get_column(self, standard_name: str) -> Optional[str]:
        """Get actual column name"""
        return self.mapping.get(standard_name)

# ========================================
# 9. SMART PREPROCESSOR
# ========================================

class SmartPreprocessor:
    """Process data based on discovered structure"""

    def __init__(self, column_mapper: ColumnMapper, evidence_mapper: EvidenceMapper):
        self.mapper = column_mapper
        self.evidence_mapper = evidence_mapper

    def parse_symptoms(self, value: Any) -> List[str]:
        """Parse symptoms safely"""
        if value is None or pd.isna(value):
            return []

        try:
            # Convert to dict
            if isinstance(value, str):
                data = ast.literal_eval(value)
            elif isinstance(value, dict):
                data = value
            else:
                return []

            # Extract positive symptoms
            symptoms = []
            for code, val in data.items():
                if val in [1, 'Y', True, 'yes', '1', 1.0, 'True', 'true']:
                    text = self.evidence_mapper.get_text(code)
                    if text:
                        symptoms.append(text)

            return symptoms

        except Exception as e:
            return []

    def parse_differential(self, value: Any) -> str:
        """Parse differential diagnosis"""
        if value is None or pd.isna(value):
            return "Not available"

        try:
            # Convert to list
            if isinstance(value, str):
                data = ast.literal_eval(value)
            elif isinstance(value, list):
                data = value
            else:
                return "Not available"

            # Format top 3
            formatted = []
            for item in data[:3]:
                if isinstance(item, dict):
                    disease = item.get('disease', item.get('condition', 'Unknown'))
                    prob = item.get('probability', 0)
                    formatted.append(f"{disease} ({prob*100:.0f}%)")
                elif isinstance(item, (list, tuple)) and len(item) >= 2:
                    disease, prob = item[0], item[1]
                    formatted.append(f"{disease} ({prob*100:.0f}%)")

            return ", ".join(formatted) if formatted else "Not available"

        except:
            return "Not available"

    def process_record(self, record: Dict, idx: int, split_name: str) -> Optional[Dict]:
        """Process single record"""
        try:
            # Get columns
            symptoms_col = self.mapper.get_column('symptoms')
            diagnosis_col = self.mapper.get_column('diagnosis')
            age_col = self.mapper.get_column('age')
            sex_col = self.mapper.get_column('sex')
            diff_col = self.mapper.get_column('differential')

            # Parse data
            symptoms = self.parse_symptoms(record.get(symptoms_col))
            symptoms_text = ", ".join(symptoms) if symptoms else "None reported"

            # Build record
            processed = {
                'patient_id': f"{split_name}_{idx}",
                'age': int(record.get(age_col, 0)) if age_col and record.get(age_col) else 0,
                'sex': str(record.get(sex_col, 'Unknown')) if sex_col else 'Unknown',
                'symptoms_text': symptoms_text,
                'symptom_count': len(symptoms),
                'pathology': str(record.get(diagnosis_col, 'Unknown')) if diagnosis_col else 'Unknown',
                'differential_diagnosis': self.parse_differential(record.get(diff_col)) if diff_col else 'Not available'
            }

            # Create combined text
            processed['combined_text'] = self._create_combined_text(processed)

            return processed

        except Exception as e:
            return None

    def _create_combined_text(self, record: Dict) -> str:
        """Create rich text for embedding"""
        parts = []

        if record['age'] and record['sex']:
            parts.append(f"Patient: {record['age']} year old {record['sex']}")

        if record['symptoms_text'] != "None reported":
            parts.append(f"Presenting symptoms: {record['symptoms_text']}")

        parts.append(f"Diagnosed condition: {record['pathology']}")

        if record['differential_diagnosis'] != "Not available":
            parts.append(f"Differential diagnosis: {record['differential_diagnosis']}")

        return ". ".join(parts) if parts else "No information available"

    def process_split(self, dataset_split, split_name: str, show_progress: bool = True) -> pd.DataFrame:
        """Process entire split"""
        logger.info(f"\n🔄 Processing {split_name}...")
        logger.info(f"   Input: {len(dataset_split):,} samples")

        processed = []
        errors = 0

        # Process with progress bar
        iterator = tqdm(
            range(len(dataset_split)),
            desc=f"Processing {split_name}",
            disable=not show_progress
        )

        for idx in iterator:
            record = dataset_split[idx]
            result = self.process_record(record, idx, split_name)

            if result:
                processed.append(result)
            else:
                errors += 1

        df = pd.DataFrame(processed)

        logger.info(f"   ✅ Processed: {len(df):,} records")
        if errors > 0:
            logger.info(f"   ⚠️ Errors: {errors} records skipped")

        return df

# ========================================
# 10. MAIN PIPELINE
# ========================================

def run_pipeline(dataset, sample_size=None, explore_sample=1000):
    """Complete preprocessing pipeline"""

    print("\n" + "="*70)
    print("🏥 DDX-PLUS SMART PREPROCESSING PIPELINE")
    print("="*70)

    # Initialize evidence mapper
    evidence_mapper = EvidenceMapper()

    # STEP 1: EXPLORE
    print("\n📊 STEP 1: Exploring Dataset Structure")
    print("─" * 70)
    explorer = DatasetExplorer(dataset)
    exploration_report = explorer.explore_all(sample_size=explore_sample)

    # STEP 2: MAP COLUMNS
    print("\n🗺️ STEP 2: Mapping Columns")
    print("─" * 70)
    column_mapper = ColumnMapper(exploration_report)

    # STEP 3: PROCESS
    print("\n⚙️ STEP 3: Processing Data")
    print("─" * 70)
    preprocessor = SmartPreprocessor(column_mapper, evidence_mapper)

    results = {}

    for split_name in dataset.keys():
        split_data = dataset[split_name]

        # Sample if needed
        if sample_size and len(split_data) > sample_size:
            logger.info(f"\n📊 Sampling {sample_size:,} from {len(split_data):,} records...")
            indices = list(range(sample_size))
            split_data = split_data.select(indices)

        # Process
        processed_df = preprocessor.process_split(split_data, split_name)
        results[split_name] = processed_df

        # Save
        output_path = PROCESSED_DIR / f"{split_name}_processed.csv"
        processed_df.to_csv(output_path, index=False)

        file_size = output_path.stat().st_size / (1024 * 1024)
        logger.info(f"   💾 Saved: {output_path.name} ({file_size:.1f} MB)")

    return results

# ========================================
# 11. SHOW STATISTICS
# ========================================

def show_statistics(processed_splits: Dict[str, pd.DataFrame]):
    """Display comprehensive statistics"""
    print("\n" + "="*70)
    print("📊 DATASET STATISTICS")
    print("="*70)

    for split_name, df in processed_splits.items():
        print(f"\n{split_name.upper()} SET:")
        print(f"  Total samples: {len(df):,}")
        print(f"  Unique diseases: {df['pathology'].nunique()}")
        print(f"  Avg symptoms: {df['symptom_count'].mean():.1f} per patient")

        # Age stats
        age_stats = df['age'][df['age'] > 0]
        if len(age_stats) > 0:
            print(f"  Age range: {age_stats.min():.0f}-{age_stats.max():.0f} years (avg: {age_stats.mean():.1f})")

        # Sex distribution
        if 'sex' in df.columns:
            sex_dist = df['sex'].value_counts()
            print(f"  Sex distribution:")
            for sex, count in sex_dist.head(3).items():
                print(f"    • {sex}: {count:,} ({count/len(df)*100:.1f}%)")

        # Top diseases
        print(f"\n  Top 5 diseases:")
        top_diseases = df['pathology'].value_counts().head(5)
        for disease, count in top_diseases.items():
            disease_name = disease[:50] + "..." if len(disease) > 50 else disease
            print(f"    • {disease_name}: {count:,} ({count/len(df)*100:.1f}%)")

    # Show samples
    print("\n" + "="*70)
    print("📋 SAMPLE PROCESSED RECORDS")
    print("="*70)

    first_split = list(processed_splits.values())[0]
    for i in range(min(2, len(first_split))):
        print(f"\n--- Sample {i+1} ---")
        sample = first_split.iloc[i]

        for col in ['patient_id', 'age', 'sex', 'symptoms_text', 'pathology']:
            if col in sample.index:
                value = str(sample[col])
                if len(value) > 100:
                    value = value[:100] + "..."
                print(f"  {col}: {value}")

# ========================================
# 12. MAIN EXECUTION
# ========================================

print("\n" + "="*70)
print("🚀 STARTING PREPROCESSING PIPELINE")
print("="*70)

# Configuration
SAMPLE_SIZE = 1000  # Set to None for full dataset
EXPLORE_SAMPLE = 1000  # Sample size for exploration

print(f"\n⚙️ Configuration:")
print(f"  Processing mode: {'Sample' if SAMPLE_SIZE else 'Full dataset'}")
if SAMPLE_SIZE:
    print(f"  Sample size: {SAMPLE_SIZE:,} records per split")
print(f"  Exploration sample: {EXPLORE_SAMPLE:,} records")

# Run pipeline
processed_splits = run_pipeline(
    dataset,
    sample_size=SAMPLE_SIZE,
    explore_sample=EXPLORE_SAMPLE
)

# Show statistics
if processed_splits:
    show_statistics(processed_splits)

    print("\n" + "="*70)
    print("✅ PREPROCESSING COMPLETE!")
    print("="*70)
    print(f"\n📁 Output locations:")
    print(f"  Processed data: {PROCESSED_DIR}")
    print(f"  Reports: {REPORTS_DIR}")

    print(f"\n📌 Files created:")
    for split in processed_splits.keys():
        file_path = PROCESSED_DIR / f"{split}_processed.csv"
        if file_path.exists():
            file_size = file_path.stat().st_size / (1024 * 1024)
            records = len(processed_splits[split])
            print(f"  ✅ {split}_processed.csv - {records:,} records ({file_size:.1f} MB)")

    print("\n✨ Ready for next step: ClinicalBERT Embeddings!")
else:
    print("\n❌ No data was processed. Check errors above.")


🚀 STARTING PREPROCESSING PIPELINE

⚙️ Configuration:
  Processing mode: Sample
  Sample size: 1,000 records per split
  Exploration sample: 1,000 records

🏥 DDX-PLUS SMART PREPROCESSING PIPELINE

📊 STEP 1: Exploring Dataset Structure
──────────────────────────────────────────────────────────────────────

🗺️ STEP 2: Mapping Columns
──────────────────────────────────────────────────────────────────────

⚙️ STEP 3: Processing Data
──────────────────────────────────────────────────────────────────────


Processing train:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing validate:   0%|          | 0/1000 [00:00<?, ?it/s]


📊 DATASET STATISTICS

TRAIN SET:
  Total samples: 1,000
  Unique diseases: 47
  Avg symptoms: 0.0 per patient
  Age range: 1-109 years (avg: 39.5)
  Sex distribution:
    • M: 500 (50.0%)
    • F: 500 (50.0%)

  Top 5 diseases:
    • Viral pharyngitis: 73 (7.3%)
    • URTI: 52 (5.2%)
    • Allergic sinusitis: 45 (4.5%)
    • Anemia: 42 (4.2%)
    • Anaphylaxis: 31 (3.1%)

TEST SET:
  Total samples: 1,000
  Unique diseases: 47
  Avg symptoms: 0.0 per patient
  Age range: 1-109 years (avg: 39.4)
  Sex distribution:
    • F: 515 (51.5%)
    • M: 485 (48.5%)

  Top 5 diseases:
    • Viral pharyngitis: 71 (7.1%)
    • URTI: 48 (4.8%)
    • Anemia: 43 (4.3%)
    • Pulmonary embolism: 36 (3.6%)
    • Panic attack: 36 (3.6%)

VALIDATE SET:
  Total samples: 1,000
  Unique diseases: 48
  Avg symptoms: 0.0 per patient
  Age range: 1-104 years (avg: 41.3)
  Sex distribution:
    • F: 531 (53.1%)
    • M: 469 (46.9%)

  Top 5 diseases:
    • URTI: 65 (6.5%)
    • Viral pharyngitis: 60 (6.0%)
    •

#ٍStep 3

In [2]:
# ========================================
# Step 2: Generate ClinicalBERT Embeddings
# ========================================

"""
✅ Load ClinicalBERT model
✅ Generate embeddings for all processed data
✅ Save embeddings efficiently
✅ Progress tracking with tqdm

What are embeddings?
- Convert text → 768-dimensional vector
- Similar texts → similar vectors
- Used for semantic search
"""

# ========================================
# 1. INSTALL & IMPORT
# ========================================

!pip install -q transformers torch pandas numpy tqdm

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModel
import pickle
import json

print("✅ Libraries imported!")

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ========================================
# 2. SETUP PATHS
# ========================================

# Mount Drive (if not already mounted)
from google.colab import drive
try:
    drive.mount('/content/drive')
except:
    print("Drive already mounted")

# Set paths
DRIVE_BASE = '/content/drive/MyDrive/DDX'  # ← Updated to match your structure
BASE_DIR = Path(DRIVE_BASE)
PROCESSED_DIR = BASE_DIR / "processed"
EMBEDDINGS_DIR = BASE_DIR / "embeddings"

# Create embeddings directory
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Directories:")
print(f"   Processed data: {PROCESSED_DIR}")
print(f"   Embeddings output: {EMBEDDINGS_DIR}")

# ========================================
# 3. VERIFY PROCESSED DATA EXISTS
# ========================================

print("\n🔍 Checking for processed data...")

processed_files = list(PROCESSED_DIR.glob("*_processed.csv"))

if not processed_files:
    print("❌ No processed files found!")
    print(f"   Expected location: {PROCESSED_DIR}")
    print("\n⚠️ Please run Step 1 (preprocessing) first!")
    raise FileNotFoundError("Processed data not found")

print(f"✅ Found {len(processed_files)} processed files:")
for file in processed_files:
    file_size = file.stat().st_size / (1024*1024)
    print(f"   • {file.name} ({file_size:.1f} MB)")

# ========================================
# 4. LOAD CLINICALBERT MODEL
# ========================================

print("\n" + "="*60)
print("🧠 LOADING CLINICALBERT MODEL")
print("="*60)

# Model details
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
print(f"\nModel: {MODEL_NAME}")
print("This model is specifically trained on clinical text!")

print("\n⏳ Downloading model... (first time only, ~400MB)")

try:
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)

    # Move model to GPU if available
    model = model.to(device)
    model.eval()  # Set to evaluation mode

    print("✅ ClinicalBERT loaded successfully!")
    print(f"   Embedding dimension: 768")
    print(f"   Max sequence length: {tokenizer.model_max_length}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

# ========================================
# 5. EMBEDDING GENERATOR CLASS
# ========================================

class ClinicalBERTEmbedder:
    """
    Generate embeddings using ClinicalBERT
    """

    def __init__(self, model, tokenizer, device, batch_size=32):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.batch_size = batch_size

        print(f"⚙️ Embedder initialized")
        print(f"   Batch size: {batch_size}")
        print(f"   Device: {device}")

    def encode_text(self, text):
        """
        Encode single text to embedding
        """
        # Tokenize
        inputs = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(self.device)

        # Generate embedding
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Use [CLS] token embedding (first token)
        embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        return embedding[0]  # Return 1D array

    def encode_batch(self, texts):
        """
        Encode batch of texts (more efficient)
        """
        # Tokenize batch
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(self.device)

        # Generate embeddings
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Extract [CLS] embeddings
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        return embeddings

    def encode_dataframe(self, df, text_column='combined_text'):
        """
        Encode entire dataframe with progress bar
        """
        print(f"\n🔄 Encoding {len(df):,} texts...")
        print(f"   Text column: {text_column}")

        embeddings = []
        num_batches = (len(df) + self.batch_size - 1) // self.batch_size

        # Process in batches
        for i in tqdm(range(0, len(df), self.batch_size),
                     total=num_batches,
                     desc="Generating embeddings"):

            batch_texts = df[text_column].iloc[i:i+self.batch_size].tolist()

            # Handle missing values
            batch_texts = [str(text) if pd.notna(text) else "" for text in batch_texts]

            # Generate embeddings
            batch_embeddings = self.encode_batch(batch_texts)
            embeddings.extend(batch_embeddings)

        embeddings_array = np.array(embeddings)

        print(f"✅ Generated embeddings shape: {embeddings_array.shape}")
        print(f"   ({len(df)} texts × 768 dimensions)")

        return embeddings_array

# ========================================
# 6. INITIALIZE EMBEDDER
# ========================================

# Set batch size based on GPU availability
BATCH_SIZE = 32 if device.type == 'cuda' else 8

embedder = ClinicalBERTEmbedder(
    model=model,
    tokenizer=tokenizer,
    device=device,
    batch_size=BATCH_SIZE
)

# ========================================
# 7. TEST EMBEDDING GENERATION
# ========================================

print("\n" + "="*60)
print("🧪 TESTING EMBEDDINGS")
print("="*60)

# Test with sample texts
test_texts = [
    "Patient: 45 year old male. Presenting symptoms: fever, cough, fatigue. Diagnosed condition: Influenza",
    "Patient: 30 year old female. Presenting symptoms: chest pain, shortness of breath. Diagnosed condition: Pneumonia",
]

print("\n📝 Test texts:")
for i, text in enumerate(test_texts, 1):
    print(f"   {i}. {text[:80]}...")

print("\n⏳ Generating test embeddings...")
test_embeddings = embedder.encode_batch(test_texts)

print(f"✅ Test successful!")
print(f"   Shape: {test_embeddings.shape}")

# Calculate similarity
from numpy.linalg import norm
similarity = np.dot(test_embeddings[0], test_embeddings[1]) / (
    norm(test_embeddings[0]) * norm(test_embeddings[1])
)
print(f"   Similarity between texts: {similarity:.3f}")
print(f"   (0 = completely different, 1 = identical)")

# ========================================
# 8. PROCESS ALL SPLITS
# ========================================

print("\n" + "="*60)
print("🚀 PROCESSING ALL DATA SPLITS")
print("="*60)

# Process each split
all_embeddings = {}
all_metadata = {}

for processed_file in processed_files:
    split_name = processed_file.stem.replace('_processed', '')

    print(f"\n{'='*60}")
    print(f"📊 Processing: {split_name.upper()}")
    print(f"{'='*60}")

    try:
        # Load processed data
        print(f"\n📂 Loading {processed_file.name}...")
        df = pd.read_csv(processed_file)
        print(f"   Loaded: {len(df):,} records")

        # Check for required column
        if 'combined_text' not in df.columns:
            print(f"❌ Error: 'combined_text' column not found!")
            print(f"   Available columns: {list(df.columns)}")
            continue

        # Generate embeddings
        embeddings = embedder.encode_dataframe(df, text_column='combined_text')

        # Save embeddings
        embeddings_file = EMBEDDINGS_DIR / f"{split_name}_embeddings.npy"
        np.save(embeddings_file, embeddings)

        file_size = embeddings_file.stat().st_size / (1024*1024)
        print(f"   💾 Saved embeddings: {embeddings_file.name} ({file_size:.1f} MB)")

        # Save metadata (for later use)
        metadata = {
            'patient_ids': df['patient_id'].tolist(),
            'pathologies': df['pathology'].tolist(),
            'symptoms': df['symptoms_text'].tolist(),
            'num_samples': len(df),
            'embedding_dim': 768,
        }

        metadata_file = EMBEDDINGS_DIR / f"{split_name}_metadata.pkl"
        with open(metadata_file, 'wb') as f:
            pickle.dump(metadata, f)

        metadata_size = metadata_file.stat().st_size / (1024*1024)
        print(f"   💾 Saved metadata: {metadata_file.name} ({metadata_size:.1f} MB)")

        # Store in memory for summary
        all_embeddings[split_name] = embeddings
        all_metadata[split_name] = metadata

        print(f"   ✅ {split_name} complete!")

    except Exception as e:
        print(f"   ❌ Error processing {split_name}: {e}")
        import traceback
        traceback.print_exc()
        continue

# ========================================
# 9. GENERATE SUMMARY
# ========================================

print("\n" + "="*60)
print("📊 EMBEDDINGS GENERATION SUMMARY")
print("="*60)

total_samples = 0
total_size = 0

for split_name in all_embeddings.keys():
    embeddings_file = EMBEDDINGS_DIR / f"{split_name}_embeddings.npy"
    metadata_file = EMBEDDINGS_DIR / f"{split_name}_metadata.pkl"

    num_samples = all_embeddings[split_name].shape[0]
    file_size = embeddings_file.stat().st_size / (1024*1024)

    print(f"\n{split_name.upper()}:")
    print(f"  Samples: {num_samples:,}")
    print(f"  Embedding shape: {all_embeddings[split_name].shape}")
    print(f"  File size: {file_size:.1f} MB")
    print(f"  Files:")
    print(f"    • {embeddings_file.name}")
    print(f"    • {metadata_file.name}")

    total_samples += num_samples
    total_size += file_size

print(f"\n{'='*60}")
print(f"TOTAL:")
print(f"  Total samples: {total_samples:,}")
print(f"  Total size: {total_size:.1f} MB")
print(f"  Location: {EMBEDDINGS_DIR}")

# ========================================
# 10. VERIFICATION TEST
# ========================================

print("\n" + "="*60)
print("🔍 VERIFICATION TEST")
print("="*60)

# Load one embedding file to verify
if all_embeddings:
    test_split = list(all_embeddings.keys())[0]

    print(f"\n✅ Testing load for: {test_split}")

    # Load from disk
    loaded_embeddings = np.load(EMBEDDINGS_DIR / f"{test_split}_embeddings.npy")
    with open(EMBEDDINGS_DIR / f"{test_split}_metadata.pkl", 'rb') as f:
        loaded_metadata = pickle.load(f)

    print(f"   Embeddings shape: {loaded_embeddings.shape}")
    print(f"   Metadata samples: {loaded_metadata['num_samples']}")
    print(f"   First patient ID: {loaded_metadata['patient_ids'][0]}")
    print(f"   First pathology: {loaded_metadata['pathologies'][0]}")

    # Verify integrity
    assert loaded_embeddings.shape[0] == loaded_metadata['num_samples']
    assert loaded_embeddings.shape[1] == 768

    print("\n✅ Verification passed! Files are valid.")

# ========================================
# 11. SAVE GENERATION INFO
# ========================================

generation_info = {
    'model_name': MODEL_NAME,
    'device': str(device),
    'batch_size': BATCH_SIZE,
    'embedding_dim': 768,
    'total_samples': total_samples,
    'splits': list(all_embeddings.keys()),
    'generation_date': pd.Timestamp.now().isoformat(),
}

info_file = EMBEDDINGS_DIR / "generation_info.json"
with open(info_file, 'w') as f:
    json.dump(generation_info, f, indent=2)

print(f"\n💾 Generation info saved: {info_file.name}")

# ========================================
# COMPLETION
# ========================================

print("\n" + "="*60)
print("✅ STEP 2 COMPLETE!")
print("="*60)
print("\n📌 What we created:")
print("   • ClinicalBERT embeddings (768-dim vectors)")
print("   • Metadata files (patient IDs, diagnoses)")
print("   • Generation info (model details)")
print(f"\n📁 All files saved in: {EMBEDDINGS_DIR}")
print("\n🎯 NEXT STEP: Build FAISS vector database for fast search!")
print("\n💡 Files ready for:")
print("   • Semantic search")
print("   • Similar case retrieval")
print("   • RAG pipeline")

✅ Libraries imported!
🖥️  Using device: cuda
   GPU: Tesla T4
   Memory: 15.8 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📁 Directories:
   Processed data: /content/drive/MyDrive/DDX/processed
   Embeddings output: /content/drive/MyDrive/DDX/embeddings

🔍 Checking for processed data...
✅ Found 3 processed files:
   • train_processed.csv (0.2 MB)
   • test_processed.csv (0.2 MB)
   • validate_processed.csv (0.2 MB)

🧠 LOADING CLINICALBERT MODEL

Model: emilyalsentzer/Bio_ClinicalBERT
This model is specifically trained on clinical text!

⏳ Downloading model... (first time only, ~400MB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

✅ Generated embeddings shape: (1000, 768)
   (1000 texts × 768 dimensions)
   💾 Saved embeddings: train_embeddings.npy (2.9 MB)
   💾 Saved metadata: train_metadata.pkl (0.0 MB)
   ✅ train complete!

📊 Processing: TEST

📂 Loading test_processed.csv...
   Loaded: 1,000 records

🔄 Encoding 1,000 texts...
   Text column: combined_text


Generating embeddings:   0%|          | 0/32 [00:00<?, ?it/s]

✅ Generated embeddings shape: (1000, 768)
   (1000 texts × 768 dimensions)
   💾 Saved embeddings: test_embeddings.npy (2.9 MB)
   💾 Saved metadata: test_metadata.pkl (0.0 MB)
   ✅ test complete!

📊 Processing: VALIDATE

📂 Loading validate_processed.csv...
   Loaded: 1,000 records

🔄 Encoding 1,000 texts...
   Text column: combined_text


Generating embeddings:   0%|          | 0/32 [00:00<?, ?it/s]

✅ Generated embeddings shape: (1000, 768)
   (1000 texts × 768 dimensions)
   💾 Saved embeddings: validate_embeddings.npy (2.9 MB)
   💾 Saved metadata: validate_metadata.pkl (0.0 MB)
   ✅ validate complete!

📊 EMBEDDINGS GENERATION SUMMARY

TRAIN:
  Samples: 1,000
  Embedding shape: (1000, 768)
  File size: 2.9 MB
  Files:
    • train_embeddings.npy
    • train_metadata.pkl

TEST:
  Samples: 1,000
  Embedding shape: (1000, 768)
  File size: 2.9 MB
  Files:
    • test_embeddings.npy
    • test_metadata.pkl

VALIDATE:
  Samples: 1,000
  Embedding shape: (1000, 768)
  File size: 2.9 MB
  Files:
    • validate_embeddings.npy
    • validate_metadata.pkl

TOTAL:
  Total samples: 3,000
  Total size: 8.8 MB
  Location: /content/drive/MyDrive/DDX/embeddings

🔍 VERIFICATION TEST

✅ Testing load for: train
   Embeddings shape: (1000, 768)
   Metadata samples: 1000
   First patient ID: train_0
   First pathology: Sarcoidosis

✅ Verification passed! Files are valid.

💾 Generation info saved: genera

#Step 4

In [3]:
# ========================================
# Step 3: Build FAISS Vector Database
# ========================================

"""
✅ Load embeddings from Step 2
✅ Build FAISS index for fast similarity search
✅ Test retrieval with sample queries
✅ Save index for later use

FAISS = Facebook AI Similarity Search
- Ultra-fast vector search
- Millions of vectors in milliseconds
- Perfect for RAG systems
"""

# ========================================
# 1. INSTALL & IMPORT
# ========================================

!pip install -q faiss-cpu numpy pandas

import faiss
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from tqdm.notebook import tqdm
import json

print("✅ Libraries imported!")

# ========================================
# 2. SETUP PATHS
# ========================================

# Mount Drive (if not already mounted)
from google.colab import drive
try:
    drive.mount('/content/drive')
except:
    print("Drive already mounted")

# Set paths
DRIVE_BASE = '/content/drive/MyDrive/DDX'
BASE_DIR = Path(DRIVE_BASE)
EMBEDDINGS_DIR = BASE_DIR / "embeddings"
FAISS_DIR = BASE_DIR / "faiss_index"
PROCESSED_DIR = BASE_DIR / "processed"

# Create FAISS directory
FAISS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Directories:")
print(f"   Embeddings: {EMBEDDINGS_DIR}")
print(f"   FAISS Index: {FAISS_DIR}")
print(f"   Processed data: {PROCESSED_DIR}")

# ========================================
# 3. VERIFY EMBEDDINGS EXIST
# ========================================

print("\n🔍 Checking for embeddings...")

embedding_files = list(EMBEDDINGS_DIR.glob("*_embeddings.npy"))

if not embedding_files:
    print("❌ No embedding files found!")
    print(f"   Expected location: {EMBEDDINGS_DIR}")
    print("\n⚠️ Please run Step 2 (Generate Embeddings) first!")
    raise FileNotFoundError("Embeddings not found")

print(f"✅ Found {len(embedding_files)} embedding files:")
for file in embedding_files:
    file_size = file.stat().st_size / (1024*1024)
    print(f"   • {file.name} ({file_size:.1f} MB)")

# ========================================
# 4. LOAD ALL EMBEDDINGS & METADATA
# ========================================

print("\n" + "="*60)
print("📥 LOADING EMBEDDINGS & METADATA")
print("="*60)

all_embeddings = []
all_metadata = []
split_info = {}

for emb_file in embedding_files:
    split_name = emb_file.stem.replace('_embeddings', '')

    print(f"\n📂 Loading: {split_name}")

    # Load embeddings
    embeddings = np.load(emb_file)
    print(f"   Embeddings: {embeddings.shape}")

    # Load metadata
    metadata_file = EMBEDDINGS_DIR / f"{split_name}_metadata.pkl"
    with open(metadata_file, 'rb') as f:
        metadata = pickle.load(f)
    print(f"   Metadata: {metadata['num_samples']} samples")

    # Store
    all_embeddings.append(embeddings)
    all_metadata.append(metadata)

    split_info[split_name] = {
        'start_idx': len(all_embeddings) - 1,
        'num_samples': len(embeddings),
        'split': split_name
    }

# Combine all embeddings
print(f"\n🔗 Combining embeddings...")
combined_embeddings = np.vstack(all_embeddings)
print(f"   Combined shape: {combined_embeddings.shape}")
print(f"   Total vectors: {combined_embeddings.shape[0]:,}")
print(f"   Dimension: {combined_embeddings.shape[1]}")

# Combine metadata
print(f"\n🔗 Combining metadata...")
combined_patient_ids = []
combined_pathologies = []
combined_symptoms = []
combined_splits = []

for i, metadata in enumerate(all_metadata):
    split_name = list(split_info.keys())[i]
    combined_patient_ids.extend(metadata['patient_ids'])
    combined_pathologies.extend(metadata['pathologies'])
    combined_symptoms.extend(metadata['symptoms'])
    combined_splits.extend([split_name] * metadata['num_samples'])

print(f"   Total records: {len(combined_patient_ids):,}")

# ========================================
# 5. BUILD FAISS INDEX
# ========================================

print("\n" + "="*60)
print("🔨 BUILDING FAISS INDEX")
print("="*60)

# Get embedding dimension
dimension = combined_embeddings.shape[1]
print(f"\n📐 Vector dimension: {dimension}")

# Normalize embeddings for cosine similarity
print(f"\n🔄 Normalizing vectors for cosine similarity...")
faiss.normalize_L2(combined_embeddings)
print(f"   ✅ Vectors normalized")

# Build FAISS index
print(f"\n🏗️ Building FAISS index...")
print(f"   Index type: IndexFlatIP (Inner Product = Cosine Similarity)")

# Create index
index = faiss.IndexFlatIP(dimension)

# Add vectors
print(f"   Adding {len(combined_embeddings):,} vectors...")
index.add(combined_embeddings)

print(f"\n✅ FAISS index built successfully!")
print(f"   Total vectors in index: {index.ntotal:,}")
print(f"   Index is trained: {index.is_trained}")

# ========================================
# 6. SAVE FAISS INDEX
# ========================================

print("\n" + "="*60)
print("💾 SAVING FAISS INDEX")
print("="*60)

# Save FAISS index
index_file = FAISS_DIR / "medical_cases.index"
faiss.write_index(index, str(index_file))
index_size = index_file.stat().st_size / (1024*1024)
print(f"\n✅ FAISS index saved: {index_file.name} ({index_size:.1f} MB)")

# Save metadata mapping
metadata_mapping = {
    'patient_ids': combined_patient_ids,
    'pathologies': combined_pathologies,
    'symptoms': combined_symptoms,
    'splits': combined_splits,
    'num_vectors': len(combined_embeddings),
    'dimension': dimension,
    'split_info': split_info,
}

mapping_file = FAISS_DIR / "metadata_mapping.pkl"
with open(mapping_file, 'wb') as f:
    pickle.dump(metadata_mapping, f)
mapping_size = mapping_file.stat().st_size / (1024*1024)
print(f"✅ Metadata mapping saved: {mapping_file.name} ({mapping_size:.1f} MB)")

# Save index info
index_info = {
    'index_type': 'IndexFlatIP',
    'dimension': dimension,
    'num_vectors': int(index.ntotal),
    'similarity_metric': 'cosine',
    'splits': list(split_info.keys()),
    'created_date': pd.Timestamp.now().isoformat(),
}

info_file = FAISS_DIR / "index_info.json"
with open(info_file, 'w') as f:
    json.dump(index_info, f, indent=2)
print(f"✅ Index info saved: {info_file.name}")

# ========================================
# 7. CREATE SEARCH FUNCTION
# ========================================

class MedicalCaseSearcher:
    """
    Search similar medical cases using FAISS
    """

    def __init__(self, index, metadata_mapping):
        self.index = index
        self.metadata = metadata_mapping
        print("✅ Medical Case Searcher initialized")
        print(f"   Index size: {self.index.ntotal:,} cases")

    def search(self, query_embedding, k=5):
        """
        Search for k most similar cases

        Args:
            query_embedding: 768-dim vector (from ClinicalBERT)
            k: number of results to return

        Returns:
            List of similar cases with scores
        """
        # Normalize query
        query_embedding = query_embedding.reshape(1, -1).astype('float32')
        faiss.normalize_L2(query_embedding)

        # Search
        scores, indices = self.index.search(query_embedding, k)

        # Get results
        results = []
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            result = {
                'rank': i + 1,
                'similarity_score': float(score),
                'patient_id': self.metadata['patient_ids'][idx],
                'pathology': self.metadata['pathologies'][idx],
                'symptoms': self.metadata['symptoms'][idx],
                'split': self.metadata['splits'][idx],
                'index': int(idx),
            }
            results.append(result)

        return results

    def print_results(self, results):
        """Pretty print search results"""
        print(f"\n{'='*60}")
        print(f"🔍 SEARCH RESULTS (Top {len(results)})")
        print(f"{'='*60}")

        for result in results:
            print(f"\n#{result['rank']} - Similarity: {result['similarity_score']:.3f} ({result['similarity_score']*100:.1f}%)")
            print(f"   Diagnosis: {result['pathology']}")
            print(f"   Symptoms: {result['symptoms'][:100]}...")
            print(f"   Patient: {result['patient_id']} ({result['split']})")

# Initialize searcher
searcher = MedicalCaseSearcher(index, metadata_mapping)

# ========================================
# 8. TEST SEARCH WITH EXAMPLES
# ========================================

print("\n" + "="*60)
print("🧪 TESTING SEARCH FUNCTIONALITY")
print("="*60)

# Test 1: Search by index (direct embedding lookup)
print("\n📊 TEST 1: Find similar cases to a specific patient")
print("-" * 60)

test_idx = 42  # Random patient
test_embedding = combined_embeddings[test_idx]

print(f"\nQuery patient:")
print(f"  ID: {combined_patient_ids[test_idx]}")
print(f"  Diagnosis: {combined_pathologies[test_idx]}")
print(f"  Symptoms: {combined_symptoms[test_idx][:100]}...")

results = searcher.search(test_embedding, k=5)
searcher.print_results(results)

# Test 2: Search by pathology
print("\n\n📊 TEST 2: Find patients with similar pathology")
print("-" * 60)

target_pathology = combined_pathologies[100]
print(f"\nSearching for cases similar to: {target_pathology}")

test_embedding_2 = combined_embeddings[100]
results_2 = searcher.search(test_embedding_2, k=5)
searcher.print_results(results_2)

# ========================================
# 9. STATISTICS & ANALYSIS
# ========================================

print("\n" + "="*60)
print("📊 INDEX STATISTICS")
print("="*60)

print(f"\n🔢 Overall:")
print(f"   Total cases: {index.ntotal:,}")
print(f"   Vector dimension: {dimension}")
print(f"   Index size: {index_size:.1f} MB")
print(f"   Metadata size: {mapping_size:.1f} MB")

print(f"\n📁 By Split:")
for split_name, info in split_info.items():
    print(f"   {split_name}: {info['num_samples']:,} cases")

print(f"\n🏥 Disease Distribution (Top 10):")
pathology_counts = pd.Series(combined_pathologies).value_counts()
for disease, count in pathology_counts.head(10).items():
    percentage = count / len(combined_pathologies) * 100
    disease_short = disease[:40] + "..." if len(disease) > 40 else disease
    print(f"   • {disease_short}: {count:,} ({percentage:.1f}%)")

# ========================================
# 10. CREATE SIMPLE QUERY INTERFACE
# ========================================

print("\n" + "="*60)
print("🎯 INTERACTIVE SEARCH INTERFACE")
print("="*60)

def search_by_symptoms(symptoms_query, top_k=5):
    """
    Search cases by symptom text
    Note: This is a simple keyword matching for demo
    For production, use ClinicalBERT to encode the query
    """
    print(f"\n🔍 Searching for: '{symptoms_query}'")
    print(f"   (Keyword-based search)")

    # Simple keyword matching
    matches = []
    query_lower = symptoms_query.lower()

    for i, symptoms in enumerate(combined_symptoms):
        if query_lower in symptoms.lower():
            matches.append(i)

    print(f"   Found {len(matches)} matches")

    if matches:
        # Get first match and search similar cases
        best_match_idx = matches[0]
        print(f"\n   Best match:")
        print(f"     Patient: {combined_patient_ids[best_match_idx]}")
        print(f"     Symptoms: {combined_symptoms[best_match_idx][:100]}...")
        print(f"     Diagnosis: {combined_pathologies[best_match_idx]}")

        # Search similar
        query_embedding = combined_embeddings[best_match_idx]
        results = searcher.search(query_embedding, k=top_k)
        searcher.print_results(results)

        return results
    else:
        print("   ❌ No matches found")
        return []

# Test interactive search
print("\n📝 Example queries:")
example_queries = [
    "fever cough",
    "chest pain",
    "headache nausea",
]

for query in example_queries:
    print(f"\n{'='*60}")
    search_by_symptoms(query, top_k=3)

# ========================================
# 11. SAVE SEARCHER FUNCTION
# ========================================

print("\n" + "="*60)
print("💾 SAVING SEARCH UTILITIES")
print("="*60)

# Save searcher code for later use
searcher_code = '''# Quick load and search utility
import faiss
import numpy as np
import pickle

# Load index
index = faiss.read_index('faiss_index/medical_cases.index')

# Load metadata
with open('faiss_index/metadata_mapping.pkl', 'rb') as f:
    metadata = pickle.load(f)

print(f"Loaded {index.ntotal:,} medical cases")

# Search function
def search_cases(query_embedding, k=5):
    query_embedding = query_embedding.reshape(1, -1).astype('float32')
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'score': float(score),
            'pathology': metadata['pathologies'][idx],
            'symptoms': metadata['symptoms'][idx],
        })
    return results
'''

utils_file = FAISS_DIR / "search_utils.py"
with open(utils_file, 'w') as f:
    f.write(searcher_code)

print(f"✅ Search utilities saved: {utils_file.name}")

# ========================================
# COMPLETION
# ========================================

print("\n" + "="*60)
print("✅ STEP 3 COMPLETE!")
print("="*60)

print("\n📌 What we created:")
print("   • FAISS vector index (fast similarity search)")
print("   • Metadata mapping (patient info)")
print("   • Search utilities (ready to use)")

print(f"\n📁 Files saved in: {FAISS_DIR}")
print("   • medical_cases.index")
print("   • metadata_mapping.pkl")
print("   • index_info.json")
print("   • search_utils.py")

print(f"\n📊 Index Statistics:")
print(f"   Total cases: {index.ntotal:,}")
print(f"   Search speed: Milliseconds for millions of vectors")
print(f"   Ready for: RAG pipeline integration")

print("\n🎯 NEXT STEP: LLM Integration (RAG)")
print("   • Connect to Claude/GPT API")
print("   • Build prompt with retrieved cases")
print("   • Generate medical responses")

print("\n💡 You can now:")
print("   1. Search similar medical cases in milliseconds")
print("   2. Retrieve relevant context for any symptoms")
print("   3. Build RAG-powered medical assistant")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.7 MB/s eta 0:00:00
✅ Libraries imported!
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📁 Directories:
   Embeddings: /content/drive/MyDrive/DDX/embeddings
   FAISS Index: /content/drive/MyDrive/DDX/faiss_index
   Processed data: /content/drive/MyDrive/DDX/processed

🔍 Checking for embeddings...
✅ Found 3 embedding files:
   • test_embeddings.npy (2.9 MB)
   • validate_embeddings.npy (2.9 MB)
   • train_embeddings.npy (2.9 MB)

📥 LOADING EMBEDDINGS & METADATA

📂 Loading: test
   Embeddings: (1000, 768)
   Metadata: 1000 samples

📂 Loading: validate
   Embeddings: (1000, 768)
   Metadata: 1000 samples

📂 Loading: train
   Embeddings: (1000, 768)
   Metadata: 1000 samples

🔗 Combining embeddings...
   Combined shape: (3000, 768)
   Total vectors: 3,000
   Dimension: 768

🔗 Combining metadata...
   Total records: 3,000

🔨 BUILDING FAISS INDEX

📐 Ve

#step 5

In [ ]:


!pip install -q google-generativeai transformers torch faiss-cpu numpy pandas

import google.generativeai as genai
import faiss
import numpy as np
import pandas as pd
import pickle
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
import json


# ========================================
# 2. SETUP PATHS
# ========================================

from google.colab import drive
try:
    drive.mount('/content/drive')
except:
    print("Drive already mounted")

DRIVE_BASE = '/content/drive/MyDrive/DDX'
BASE_DIR = Path(DRIVE_BASE)
FAISS_DIR = BASE_DIR / "faiss_index"
EMBEDDINGS_DIR = BASE_DIR / "embeddings"

print(f"\n📁 Directories:")
print(f"   FAISS: {FAISS_DIR}")
print(f"   Embeddings: {EMBEDDINGS_DIR}")

# ========================================
# 3. SETUP GOOGLE GEMINI API (FREE)
# ========================================

print("\n" + "="*60)
print("🔑 SETUP GOOGLE GEMINI API (FREE)")
print("="*60)

print("\n📝 Get your FREE API key:")
print("   1. Go to: https://makersuite.google.com/app/apikey")
print("   2. Click 'Create API Key'")
print("   3. Copy the key")
print("\n   (It's FREE - no credit card needed!)")

# Enter API key
import getpass
GOOGLE_API_KEY = getpass.getpass("\n🔑 Paste your Gemini API Key: ")

# Configure Gemini
genai.configure(api_key=GOOGLE_API_KEY)

# Test API
try:
    model_test = genai.GenerativeModel('gemini-2.5-flash')
    response_test = model_test.generate_content("Say hello in one word")
    print(f"\n✅ Gemini API connected successfully!")
    print(f"   Test response: {response_test.text}")
    print(f"   Model: gemini-1.5-flash (FREE & FAST)")
except Exception as e:
    print(f"\n❌ API Error: {e}")
    print("\n💡 Make sure:")
    print("   1. API key is correct")
    print("   2. You have internet connection")
    print("   3. API is enabled at https://makersuite.google.com")

# ========================================
# 4. LOAD CLINICAL BERT
# ========================================

print("\n" + "="*60)
print("🧠 LOADING CLINICALBERT")
print("="*60)

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

print(f"\n⏳ Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)
bert_model.eval()

print(f"✅ ClinicalBERT loaded on {device}")

# ========================================
# 5. LOAD FAISS INDEX
# ========================================

print("\n" + "="*60)
print("📚 LOADING FAISS INDEX")
print("="*60)

# Load FAISS index
index_file = FAISS_DIR / "medical_cases.index"
index = faiss.read_index(str(index_file))
print(f"✅ FAISS index loaded: {index.ntotal:,} cases")

# Load metadata
metadata_file = FAISS_DIR / "metadata_mapping.pkl"
with open(metadata_file, 'rb') as f:
    metadata_mapping = pickle.load(f)
print(f"✅ Metadata loaded: {metadata_mapping['num_vectors']:,} records")

# ========================================
# 6. CREATE RAG MEDICAL ASSISTANT
# ========================================

class MedicalRAGAssistant:
    """
    Complete RAG-powered Medical Assistant with Google Gemini
    """

    def __init__(self, bert_model, tokenizer, faiss_index, metadata, gemini_api_key, device):
        self.bert_model = bert_model
        self.tokenizer = tokenizer
        self.index = faiss_index
        self.metadata = metadata
        self.device = device

        # Initialize Gemini
        genai.configure(api_key=gemini_api_key)
        # Use the new model name: gemini-1.5-flash or gemini-1.5-pro
        self.gemini_model = genai.GenerativeModel('gemini-2.5-flash')

        print("✅ Medical RAG Assistant initialized")
        print(f"   Knowledge base: {self.index.ntotal:,} medical cases")
        print(f"   LLM: Google Gemini 1.5 Flash (FREE & FAST)")
        print(f"   Encoder: ClinicalBERT")

    def encode_query(self, text):
        """Encode user query using ClinicalBERT"""
        inputs = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.bert_model(**inputs)

        embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        return embedding[0]

    def retrieve_similar_cases(self, query_embedding, k=5):
        """Search FAISS for similar cases"""
        query_embedding = query_embedding.reshape(1, -1).astype('float32')
        faiss.normalize_L2(query_embedding)

        scores, indices = self.index.search(query_embedding, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            # Get symptoms text - handle if empty
            symptoms = self.metadata['symptoms'][idx]
            if not symptoms or symptoms == "None reported" or symptoms.strip() == "":
                symptoms = f"Patient with {self.metadata['pathologies'][idx]}"

            results.append({
                'similarity': float(score),
                'pathology': self.metadata['pathologies'][idx],
                'symptoms': symptoms,
                'patient_id': self.metadata['patient_ids'][idx],
            })

        return results

    def build_rag_prompt(self, user_symptoms, retrieved_cases):
        """Build prompt with retrieved context"""

        # Format retrieved cases
        context = "SIMILAR MEDICAL CASES FROM DATABASE:\n\n"

        for i, case in enumerate(retrieved_cases, 1):
            context += f"Case {i} (Similarity: {case['similarity']*100:.1f}%):\n"
            context += f"- Diagnosis: {case['pathology']}\n"
            context += f"- Symptoms: {case['symptoms']}\n\n"

        # Build full prompt
        prompt = f"""You are a medical AI assistant analyzing patient symptoms.

PATIENT'S SYMPTOMS:
{user_symptoms}

{context}

Based on these similar cases from our medical database, provide a comprehensive analysis:

1. **Most Likely Diagnosis**: What condition best matches these symptoms?

2. **Reasoning**: Why does this diagnosis fit? Reference the similar cases above.

3. **Key Symptoms Match**: Which symptoms are most indicative?

4. **Recommended Actions**: What should the patient do next?

5. **Warning Signs**: When should they seek immediate medical attention?

Keep your response clear, professional, and evidence-based. Use the similar cases as supporting evidence."""

        return prompt

    def generate_response(self, prompt):
        """Generate response using Gemini"""
        try:
            # Configure generation settings
            clean_prompt = prompt.encode('utf-8', 'ignore').decode('utf-8')
            generation_config = {
                'temperature': 0.3,  # Lower = more focused
                'top_p': 0.8,
                'top_k': 40,
                'max_output_tokens': 1024,
            }

            # Generate response
            response = self.gemini_model.generate_content(
                prompt,
                generation_config=generation_config
            )

            return response.text

        except Exception as e:
            return f"Error generating response: {e}"

    def add_medical_disclaimer(self, response):
        """Add safety disclaimer"""
        disclaimer = """

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️ IMPORTANT MEDICAL DISCLAIMER

This response is generated by AI based on pattern matching with medical cases.
It is NOT a substitute for professional medical advice, diagnosis, or treatment.

✋ Always seek the advice of your physician or other qualified health provider
   with any questions you may have regarding a medical condition.

🚨 EMERGENCY: If you are experiencing a medical emergency, call emergency
   services immediately (911, 123, or your local emergency number).

This AI assistant is for informational purposes only.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"""

        return response + disclaimer

    def chat(self, user_symptoms, top_k=5, verbose=True):
        """
        Main chat function - complete RAG pipeline
        """
        if verbose:
            print("\n" + "="*70)
            print("🏥 MEDICAL RAG ASSISTANT (Powered by Google Gemini)")
            print("="*70)
            print(f"\n👤 Patient Query: {user_symptoms}")

        # Step 1: Encode query
        if verbose:
            print("\n🔄 Step 1: Encoding symptoms with ClinicalBERT...")
        query_embedding = self.encode_query(user_symptoms)
        if verbose:
            print("   ✅ Symptoms encoded to 768-dim vector")

        # Step 2: Retrieve similar cases
        if verbose:
            print(f"\n🔍 Step 2: Searching {self.index.ntotal:,} medical cases...")
        retrieved_cases = self.retrieve_similar_cases(query_embedding, k=top_k)
        if verbose:
            print(f"   ✅ Found {len(retrieved_cases)} similar cases")
            print("\n📋 Top Similar Cases:")
            for i, case in enumerate(retrieved_cases[:3], 1):
                symptoms_preview = case['symptoms'][:60] + "..." if len(case['symptoms']) > 60 else case['symptoms']
                print(f"   {i}. {case['pathology']}")
                print(f"      Similarity: {case['similarity']*100:.1f}%")
                print(f"      Symptoms: {symptoms_preview}")

        # Step 3: Build prompt
        if verbose:
            print("\n📝 Step 3: Building RAG prompt with context...")
        prompt = self.build_rag_prompt(user_symptoms, retrieved_cases)
        if verbose:
            print("   ✅ Prompt ready")

        # Step 4: Generate response
        if verbose:
            print("\n🤖 Step 4: Generating response with Gemini Pro...")
        response = self.generate_response(prompt)
        if verbose:
            print("   ✅ Response generated")

        # Step 5: Add disclaimer
        final_response = self.add_medical_disclaimer(response)

        if verbose:
            print("\n" + "="*70)
            print("💬 AI ASSISTANT RESPONSE:")
            print("="*70)
            print(final_response)
            print("\n" + "="*70)

        return {
            'query': user_symptoms,
            'retrieved_cases': retrieved_cases,
            'response': final_response,
            'raw_response': response,
        }

# ========================================
# 7. INITIALIZE ASSISTANT
# ========================================

print("\n" + "="*60)
print("🚀 INITIALIZING MEDICAL RAG ASSISTANT")
print("="*60)

assistant = MedicalRAGAssistant(
    bert_model=bert_model,
    tokenizer=tokenizer,
    faiss_index=index,
    metadata=metadata_mapping,
    gemini_api_key=GOOGLE_API_KEY,
    device=device
)

# ========================================
# 8. TEST WITH EXAMPLES
# ========================================

print("\n" + "="*60)
print("🧪 TESTING WITH EXAMPLE QUERIES")
print("="*60)

# Test cases
test_queries = [
    "I have high fever, severe cough, and body aches for 3 days",
    "Sudden chest pain with difficulty breathing and sweating",
    "Persistent headache with nausea and sensitivity to bright light",
]

print(f"\n📝 Running {len(test_queries)} test queries...")
print("   (Each query takes ~10-15 seconds)")

test_results = []

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*70}")
    print(f"TEST QUERY {i}/{len(test_queries)}")
    print(f"{'='*70}")

    result = assistant.chat(query, top_k=5, verbose=True)
    test_results.append(result)

    # Save result
    result_file = BASE_DIR / f"test_result_{i}.json"
    with open(result_file, 'w') as f:
        json_result = {
            'query': result['query'],
            'response': result['raw_response'],
            'top_cases': [
                {
                    'pathology': c['pathology'],
                    'similarity': f"{c['similarity']*100:.1f}%"
                }
                for c in result['retrieved_cases'][:3]
            ]
        }
        json.dump(json_result, f, indent=2)

    print(f"\n💾 Result saved to: {result_file.name}")

# ========================================
# 9. INTERACTIVE MODE
# ========================================

print("\n" + "="*60)
print("💬 INTERACTIVE CHAT MODE")
print("="*60)

def interactive_chat():
    """Interactive chat loop"""
    print("\n🏥 Welcome to Medical RAG Assistant!")
    print("   Powered by ClinicalBERT + FAISS + Google Gemini")
    print("\n💡 Describe your symptoms or medical concerns.")
    print("   Type 'quit', 'exit', or 'q' to stop.\n")

    conversation_history = []

    while True:
        try:
            user_input = input("👤 You: ").strip()

            if user_input.lower() in ['quit', 'exit', 'q', 'stop']:
                print("\n👋 Thank you for using Medical RAG Assistant!")
                print(f"   Total queries in session: {len(conversation_history)}")
                break

            if not user_input:
                continue

            # Get response
            print("\n⏳ Processing... (this may take 10-15 seconds)")
            result = assistant.chat(user_input, top_k=5, verbose=False)

            print(f"\n🤖 Assistant:\n")
            print(result['response'])
            print("\n" + "-"*70 + "\n")

            # Save to history
            conversation_history.append({
                'user': user_input,
                'assistant': result['raw_response'],
                'top_diagnosis': result['retrieved_cases'][0]['pathology'] if result['retrieved_cases'] else 'N/A'
            })

        except KeyboardInterrupt:
            print("\n\n👋 Chat interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")
            continue

# Start interactive mode
print("\n🎯 Starting interactive chat in 3 seconds...")
print("   (Press Ctrl+C to skip if you don't want interactive mode)")

import time
try:
    time.sleep(3)
    interactive_chat()
except KeyboardInterrupt:
    print("\n⏭️ Skipped interactive mode")

# ========================================
# 10. SAVE SYSTEM INFO
# ========================================

print("\n" + "="*60)
print("💾 SAVING SYSTEM INFORMATION")
print("="*60)

system_info = {
    'bert_model': MODEL_NAME,
    'llm_model': 'Google Gemini Pro',
    'index_size': int(index.ntotal),
    'created_date': pd.Timestamp.now().isoformat(),
    'components': {
        'encoder': 'ClinicalBERT (768-dim)',
        'vector_db': 'FAISS IndexFlatIP',
        'llm': 'Google Gemini Pro (FREE)',
        'safety': 'Medical disclaimer included'
    },
    'test_queries_run': len(test_results),
    'api_cost': 'FREE (Gemini API)'
}

system_file = BASE_DIR / "rag_system_info.json"
with open(system_file, 'w') as f:
    json.dump(system_info, f, indent=2)

print(f"✅ System info saved: {system_file.name}")

# Print summary
print("\n" + "="*60)
print("📊 SYSTEM SUMMARY")
print("="*60)
print(f"\n✅ Components Initialized:")
print(f"   • ClinicalBERT: ✅ Loaded")
print(f"   • FAISS Index: ✅ {index.ntotal:,} cases")
print(f"   • Google Gemini: ✅ Connected (FREE)")
print(f"   • Medical Disclaimer: ✅ Included")

print(f"\n📁 Generated Files:")
print(f"   • Test results: {len(test_results)} queries saved")
print(f"   • System info: rag_system_info.json")
print(f"   • Location: {BASE_DIR}")

# ========================================
# COMPLETION
# ========================================

print("\n" + "="*60)
print("✅ STEP 4 COMPLETE - RAG SYSTEM READY!")
print("="*60)

print("\n🎉 Congratulations! Your Medical RAG Assistant is fully functional!")

print("\n🎯 What you can do now:")
print("\n1. Quick Query:")
print("   result = assistant.chat('I have fever and cough')")

print("\n2. Interactive Chat:")
print("   interactive_chat()")

print("\n3. Custom Query:")
print("   result = assistant.chat(")
print("       'your symptoms here',")
print("       top_k=10,  # retrieve more cases")
print("       verbose=True  # show details")
print("   )")

print("\n💰 Cost: $0 (100% FREE with Gemini API)")

print("\n📌 Key Features:")
print("   ✅ Medical knowledge: 3,000+ cases")
print("   ✅ Semantic search: ClinicalBERT embeddings")
print("   ✅ AI responses: Google Gemini Pro")
print("   ✅ Safety: Medical disclaimers included")
print("   ✅ Cost: FREE!")

print("\n🚀 Your NMP Medical Assistant is READY!")
print("\n💡 Next: Build a UI (Gradio/Streamlit) for easy access!")

#6

In [9]:
# ========================================
# Step 5: Beautiful Gradio UI - Medical RAG Assistant
# ========================================

"""
✅ Beautiful web interface
✅ Chat-like interface
✅ Multi-turn conversations
✅ Show similar cases
✅ Export chat history
✅ Mobile-friendly
✅ Share publicly (optional)

Run this after Step 4 (LLM Integration)
"""

# ========================================
# 1. INSTALL GRADIO
# ========================================

!pip install -q gradio

import gradio as gr
import json
from datetime import datetime

print("✅ Gradio installed!")

# ========================================
# 2. VERIFY ASSISTANT EXISTS
# ========================================

try:
    # Check if assistant is already initialized from Step 4
    test_query = assistant.encode_query("test")
    print("✅ Medical RAG Assistant found!")
    print(f"   Knowledge base: {assistant.index.ntotal:,} cases")
except NameError:
    print("❌ Assistant not found!")
    print("\n⚠️ Please run Step 4 (LLM Integration) first!")
    print("   The 'assistant' variable needs to be initialized.")
    raise

# ========================================
# 3. CREATE CHAT INTERFACE
# ========================================

class GradioMedicalChat:
    """
    Gradio interface wrapper for Medical RAG Assistant
    """

    def __init__(self, assistant):
        self.assistant = assistant
        self.chat_history = []
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")

    def chat_function(self, message, history):
        """
        Main chat function for Gradio

        Args:
            message: User's current message
            history: List of [user_msg, bot_msg] pairs

        Returns:
            Updated history
        """
        if not message or message.strip() == "":
            return history

        try:
            # Get response from assistant
            result = self.assistant.chat(
                message,
                top_k=5,
                verbose=False  # Don't print details in UI
            )

            # Format response with similar cases
            bot_response = self._format_response(result)

            # Save to history
            self.chat_history.append({
                'user': message,
                'assistant': result['raw_response'],
                'retrieved_cases': result['retrieved_cases'],
                'timestamp': datetime.now().isoformat()
            })

            # Return updated history
            return history + [[message, bot_response]]

        except Exception as e:
            error_response = f"⚠️ Sorry, I encountered an error: {str(e)}\n\nPlease try rephrasing your question."
            return history + [[message, error_response]]

    def _format_response(self, result):
        """Format response with similar cases info"""
        response = result['response']

        # Add similar cases summary at the top
        cases_summary = "\n\n---\n\n**📊 Most Similar Cases Found:**\n\n"
        for i, case in enumerate(result['retrieved_cases'][:3], 1):
            cases_summary += f"{i}. **{case['pathology']}** (Similarity: {case['similarity']*100:.1f}%)\n"

        # Combine
        full_response = cases_summary + "\n---\n\n" + response

        return full_response

    def export_history(self):
        """Export chat history as JSON"""
        if not self.chat_history:
            return "No chat history to export."

        filename = f"chat_history_{self.session_id}.json"
        filepath = f"/content/drive/MyDrive/DDX/{filename}"

        with open(filepath, 'w') as f:
            json.dump(self.chat_history, f, indent=2)

        return f"✅ Chat history exported to:\n{filepath}"

    def clear_history(self):
        """Clear chat history"""
        self.chat_history = []
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        return "✅ Chat history cleared!"

# Initialize chat wrapper
gradio_chat = GradioMedicalChat(assistant)

# ========================================
# 4. CREATE GRADIO INTERFACE
# ========================================

# Custom CSS for better styling
custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
}

.medical-header {
    text-align: center;
    padding: 20px;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    border-radius: 10px;
    margin-bottom: 20px;
}

.warning-box {
    background-color: #fff3cd;
    border: 2px solid #ffc107;
    border-radius: 8px;
    padding: 15px;
    margin: 10px 0;
}

.stats-box {
    background-color: #e7f3ff;
    border: 2px solid #2196F3;
    border-radius: 8px;
    padding: 15px;
    margin: 10px 0;
}
"""

# Create interface
with gr.Blocks(css=custom_css, title="Medical RAG Assistant", theme=gr.themes.Soft()) as demo:

    # Header
    gr.HTML("""
        <div class="medical-header">
            <h1>🏥 Medical RAG Assistant</h1>
            <p>AI-Powered Medical Symptom Analysis</p>
            <p style="font-size: 14px; opacity: 0.9;">
                Powered by ClinicalBERT + FAISS + Google Gemini 2.0
            </p>
        </div>
    """)

    # Warning Banner
    gr.HTML("""
        <div class="warning-box">
            <h3>⚠️ Important Medical Disclaimer</h3>
            <p>
                This is an AI assistant for <strong>informational purposes only</strong>.
                It is <strong>NOT</strong> a substitute for professional medical advice.
                Always consult with a qualified healthcare provider for proper diagnosis and treatment.
            </p>
            <p>
                <strong>🚨 Emergency:</strong> If experiencing a medical emergency,
                call emergency services immediately!
            </p>
        </div>
    """)

    # Stats
    gr.HTML(f"""
        <div class="stats-box">
            <h3>📊 System Information</h3>
            <ul>
                <li><strong>Knowledge Base:</strong> {assistant.index.ntotal:,} medical cases</li>
                <li><strong>AI Model:</strong> Google Gemini 2.0 Flash</li>
                <li><strong>Search Engine:</strong> FAISS Vector Database</li>
                <li><strong>Medical Encoder:</strong> ClinicalBERT</li>
            </ul>
        </div>
    """)

    # Main Chat Interface
    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                height=500,
                label="Chat with Medical Assistant",
                show_label=True,
                bubble_full_width=False,
                avatar_images=(None, "https://em-content.zobj.net/source/twitter/348/health-worker_1f9d1-200d-2695-fe0f.png")
            )

            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Describe your symptoms here... (e.g., 'I have fever and cough for 3 days')",
                    label="Your Symptoms",
                    lines=3,
                    scale=4
                )
                send_btn = gr.Button("Send 🚀", variant="primary", scale=1)

            with gr.Row():
                clear_btn = gr.Button("Clear Chat 🗑️", size="sm")
                export_btn = gr.Button("Export History 💾", size="sm")

        # Sidebar with examples and info
        with gr.Column(scale=1):
            gr.Markdown("### 💡 Example Queries")

            example_1 = gr.Button("🤒 Fever & Cough", size="sm")
            example_2 = gr.Button("💔 Chest Pain", size="sm")
            example_3 = gr.Button("🤕 Headache & Nausea", size="sm")
            example_4 = gr.Button("😰 Fatigue & Weakness", size="sm")

            gr.Markdown("---")
            gr.Markdown("""
                ### ℹ️ How to Use

                1. **Describe your symptoms** clearly
                2. **Include duration** (how long?)
                3. **Mention severity** (mild/severe?)
                4. **Add relevant details**

                The AI will:
                - Search similar cases
                - Provide possible diagnosis
                - Suggest next steps
            """)

    # Export status
    export_status = gr.Textbox(label="Export Status", visible=False)

    # ========================================
    # 5. EVENT HANDLERS
    # ========================================

    # Send button click
    send_btn.click(
        fn=gradio_chat.chat_function,
        inputs=[msg, chatbot],
        outputs=chatbot
    ).then(
        fn=lambda: "",  # Clear input after sending
        inputs=None,
        outputs=msg
    )

    # Enter key press
    msg.submit(
        fn=gradio_chat.chat_function,
        inputs=[msg, chatbot],
        outputs=chatbot
    ).then(
        fn=lambda: "",
        inputs=None,
        outputs=msg
    )

    # Clear chat
    clear_btn.click(
        fn=lambda: ([], gradio_chat.clear_history()),
        inputs=None,
        outputs=[chatbot, export_status]
    )

    # Export history
    export_btn.click(
        fn=gradio_chat.export_history,
        inputs=None,
        outputs=export_status
    ).then(
        fn=lambda x: gr.update(visible=True),
        inputs=export_status,
        outputs=export_status
    )

    # Example buttons
    example_1.click(
        fn=lambda: "I have high fever, severe cough, and body aches for 3 days",
        inputs=None,
        outputs=msg
    )

    example_2.click(
        fn=lambda: "Sudden chest pain with difficulty breathing and sweating",
        inputs=None,
        outputs=msg
    )

    example_3.click(
        fn=lambda: "Persistent headache with nausea and sensitivity to light",
        inputs=None,
        outputs=msg
    )

    example_4.click(
        fn=lambda: "Extreme fatigue and weakness for the past week",
        inputs=None,
        outputs=msg
    )

    # Footer
    gr.HTML("""
        <div style="text-align: center; margin-top: 30px; padding: 20px; background-color: #f8f9fa; border-radius: 8px;">
            <p style="color: #666; font-size: 14px;">
                <strong>NMP Medical Assistant</strong> - AI-Powered Healthcare Support<br>
                Built with ClinicalBERT, FAISS, and Google Gemini 2.0<br>
                <em>For educational and informational purposes only</em>
            </p>
        </div>
    """)

# ========================================
# 6. LAUNCH INTERFACE
# ========================================

print("\n" + "="*60)
print("🚀 LAUNCHING GRADIO INTERFACE")
print("="*60)

print("\n📱 Interface Features:")
print("   ✅ Chat-like interface")
print("   ✅ Example queries")
print("   ✅ Export chat history")
print("   ✅ Mobile-friendly")
print("   ✅ Professional medical disclaimer")

print("\n🌐 Launching in 3 seconds...")

import time
time.sleep(3)

# Launch options
demo.launch(
    share=True,           # Creates public link (optional)
    debug=False,          # Set to True for debugging
    show_error=True,      # Show errors in UI
    quiet=False,          # Show startup messages
    inline=False,         # Open in new tab
)

print("\n✅ Interface launched!")
print("\n💡 Tips:")
print("   • The interface will open automatically")
print("   • Use 'share=True' to get a public link")
print("   • Chat history is saved automatically")
print("   • Use Export button to save conversations")

print("\n🎉 Your Medical RAG Assistant UI is ready!")

✅ Gradio installed!
✅ Medical RAG Assistant found!
   Knowledge base: 3,000 cases


/tmp/ipython-input-3152839443.py:171: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="Medical RAG Assistant", theme=gr.themes.Soft()) as demo:
/tmp/ipython-input-3152839443.py:171: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="Medical RAG Assistant", theme=gr.themes.Soft()) as demo:
/tmp/ipython-input-3152839443.py:216: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipython-input-3152839443.py:216: DeprecationWarn


🚀 LAUNCHING GRADIO INTERFACE

📱 Interface Features:
   ✅ Chat-like interface
   ✅ Example queries
   ✅ Export chat history
   ✅ Mobile-friendly
   ✅ Professional medical disclaimer

🌐 Launching in 3 seconds...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ef26e48079e18c79f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)

✅ Interface launched!

💡 Tips:
   • The interface will open automatically
   • Use 'share=True' to get a public link
   • Chat history is saved automatically
   • Use Export button to save conversations

🎉 Your Medical RAG Assistant UI is ready!
